# Finetune YOLO26n on WIDERFACE Dataset
The dataset can be found at: https://www.kaggle.com/datasets/lylmsc/wider-face-for-yolo-training

This notebook finetunes the YOLO26n model on face detection task. 

This training is copied with modifications (in augmentation) from Nyakorare: https://www.kaggle.com/datasets/lylmsc/wider-face-for-yolo-training

In [ ]:
!pip install ultralytics

In [ ]:
import os
import glob
import shutil
import random
from pathlib import Path

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
from ultralytics import YOLO
print('Libraies imported')

In [ ]:
# Paths and directories
ROOT_DIR = "/kaggle/input/datasets/lylmsc/wider-face-for-yolo-training"
IMAGE_DIR = f"{ROOT_DIR}/images"
LABEL_DIR = f"{ROOT_DIR}/labels"

OUTPUT_DIR = "/kaggle/working/face_detection"
TRAIN_IMG_DIR = f"{OUTPUT_DIR}/images/train"
VAL_IMG_DIR = f"{OUTPUT_DIR}/images/val"
TRAIN_LABEL_DIR = f"{OUTPUT_DIR}/labels/train"
VAL_LABEL_DIR = f"{OUTPUT_DIR}/labels/val"

RESULTS_DIR = f"{OUTPUT_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

for d in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LABEL_DIR, VAL_LABEL_DIR]:
    os.makedirs(d, exist_ok=True)

# Collect image files and split (80/20)
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]
train_files, val_files = train_test_split(image_files, test_size=0.2, random_state=42)

def copy_files(file_list, src_img_dir, src_label_dir, dest_img_dir, dest_label_dir):
    count_img = 0
    count_label = 0
    for file in tqdm(file_list, desc="Copying files"):
        base_name = os.path.splitext(file)[0]
        img_src = os.path.join(src_img_dir, file)
        img_dst = os.path.join(dest_img_dir, file)
        if os.path.exists(img_src):
            shutil.copy2(img_src, img_dst)
            count_img += 1
        label_src = os.path.join(src_label_dir, f"{base_name}.txt")
        label_dst = os.path.join(dest_label_dir, f"{base_name}.txt")
        if os.path.exists(label_src):
            shutil.copy2(label_src, label_dst)
            count_label += 1
    return count_img, count_label

train_img_count, train_label_count = copy_files(train_files, IMAGE_DIR, LABEL_DIR, TRAIN_IMG_DIR, TRAIN_LABEL_DIR)
val_img_count, val_label_count = copy_files(val_files, IMAGE_DIR, LABEL_DIR, VAL_IMG_DIR, VAL_LABEL_DIR)

print(f"Total images: {len(image_files)}")
print(f"Train: {len(train_files)} ({len(train_files)/len(image_files):.1%}), Validation: {len(val_files)} ({len(val_files)/len(image_files):.1%})")
print(f"Copied {train_img_count} train images and {train_label_count} train labels")
print(f"Copied {val_img_count} val images and {val_label_count} val labels")


Copying files:   0%|          | 0/10304 [00:00<?, ?it/s]

Copying files:   0%|          | 0/2576 [00:00<?, ?it/s]

Total images: 12880
Train: 10304 (80.0%), Validation: 2576 (20.0%)
Copied 10304 train images and 10304 train labels
Copied 2576 val images and 2576 val labels


In [ ]:
data_yaml = {
    'path': OUTPUT_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': ['face']
}
with open(f"{OUTPUT_DIR}/data.yaml", 'w') as f:
    yaml.dump(data_yaml, f)
print(f"Created config: {OUTPUT_DIR}/data.yaml")

Created config: /kaggle/working/face_detection/data.yaml


In [ ]:
hyp_yaml = {
    'hsv_h': 0.015, 
    'hsv_s': 0.8,     # Increased: Simulates washed-out colors (fog) or extreme color shifts
    'hsv_v': 0.8,     # Massively Increased: Forces the model to learn in near-pitch darkness and blinding glare
    # Geometry & Camera Shake (Simulating Wind/Storms)
    'degrees': 15.0,  # Increased: Simulates camera sway in heavy wind
    'translate': 0.2, # Increased: Simulates violent camera jitter
    'scale': 0.6,     
    'shear': 2.0,     # Added: Slight shearing mimics rain distortion through glass
    'perspective': 0.001, 
    
    # Standard Face Constraints
    'flipud': 0.0,    # Keep at 0: People don't walk upside down, even in a storm
    'fliplr': 0.5,    
    
    # Structural Destruction (Simulating Occlusion & Fog)
    'mosaic': 1.0,    
    'mixup': 0.2,     # Increased to 20%: Mixup blends images together. This creates a ghosting effect that mimics heavy fog perfectly.
    'copy_paste': 0.0,
    'erasing': 0.4    # Added (if your YOLO version supports it): Randomly blacks out chunks of the image, simulating mud or large raindrops on the lens.
}

with open(f"{OUTPUT_DIR}/hyp.yaml", 'w') as f:
    yaml.dump(hyp_yaml, f)
print(f"Saved extreme weather config: {OUTPUT_DIR}/hyp.yaml")

Saved extreme weather config: /kaggle/working/face_detection/hyp.yaml


In [ ]:
model = YOLO('yolo26n.pt')

In [ ]:
results = model.train(
    data=f"{OUTPUT_DIR}/data.yaml",
    epochs=50,
    imgsz=640,
    device=0,
    batch=16,
    workers=4,
    save_period=1,
    project=RESULTS_DIR,
    name="train",
    patience=20,
    optimizer='AdamW',
    lr0=0.002,
    lrf=0.01,
    hsv_h=0.015,
    hsv_s=0.8,
    hsv_v=0.8,
    degrees=15.0,
    translate=0.2,
    scale=0.6,
    shear=2.0,
    perspective=0.001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.0,
    erasing=0.4,
    pretrained=True,
    verbose=False
)

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/face_detection/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.8, hsv_v=0.8, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pa